## Project: Mapping the Potential Destructive Power of Wildfires Using Machine Learning

### Module 1: *Data Processing*

---
### Contents  
- 1. *Data Loading and Exploration* 
- 2. *Data Cleaning and Merging*
- 3. *Data Imputation*
- 3. *File Export*
---
### Notes

**Clean and process daily weather readings from California CIMIS irrigation stations (2018-01-01 to 2024-12-31). Integrate wildfire impact data from the same period, including structures damaged/destroyed and acres burned. 

---
### Inputs
*Raw Data*
- Weather Data - `CIMISdata.csv` 
- Fire Damage Data - `fire_damages.csv` (Wildfire damage data cleaned in Appendix A)
- Miscellaneous Data - `i04_CIMIS_Weather_Stations.csv`, `pop_data`, `mean_income` 
---
### Outputs  
- `weasther_fire_pop_income.csv` Cleaned weather dataset merged with fire damage severity 
---
### User Created Dependencies  

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

# Convenience function to impute a column with the median
from src.data_utils import impute_median_data

# Show basic facts about a dataframe
from src.data_utils import basic_explore

# basic health checks after a merge
from src.data_utils import post_merge_check

# function to identify the number of missing dates in the dataset
from src.data_utils import identify_missing_station_dates

# Function to print a grid of kde plots in consistent format, adjusts columns and rows accordingly
from src.plot_utils import grid_kde

---
### Third Party Dependencies

In [2]:
# Data handling
import pandas as pd
import numpy as np
import geopandas as gpd
import datetime as dt

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns
from shapely.geometry import Point

# Preprocessing and modeling utilities
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

## Global Constants

In [3]:
# first day to analyze in weather dataset
FIRST_DATE = pd.to_datetime('2018-01-01').date()

# last day to analyze in weather dataset
LAST_DATE = pd.to_datetime('2024-12-31').date()

---

## 1. Data Loading and Exploration

### 1.1 CIMIS Historical Weather Data - `CIMISdata.csv`

This dataset contains daily weather measurements from CIMIS stations across California. Variables include temperature, precipitation, humidity, solar radiation, and wind. 

Data sourced directly from:
[California CIMIS Data](https://cimis.water.ca.gov/WSNReportCriteria.aspx)

In [4]:
weather = pd.read_csv("../data/raw/CIMISdata.csv")
weather.loc[:, 'Date'] = pd.to_datetime(weather['Date'], format='mixed').dt.date
basic_explore(weather)
weather.head(3)

Rows:  351978  Columns:  18
Duplicates  1314
Total NA values:  73723  of  6335604 datapoints


,Stn Id,Stn Name,CIMIS Region,Date,ETo (in),Precip (in),Sol Rad (Ly/day),Avg Vap Pres (mBars),Max Air Temp (F),Min Air Temp (F),Avg Air Temp (F),Max Rel Hum (%),Min Rel Hum (%),Avg Rel Hum (%),Dew Point (F),Avg Wind Speed (mph),Wind Run (miles),Avg Soil Temp (F)
0,2,FivePoints,San Joaquin Valley,2018-01-01,0.06,0.0,219.0,7.3,63.4,35.3,47.8,82.0,46.0,65.0,36.6,3.3,78.3,51.1
1,2,FivePoints,San Joaquin Valley,2018-01-02,0.04,0.0,127.0,7.4,59.8,37.7,47.2,80.0,52.0,67.0,36.7,3.1,74.5,51.3
2,2,FivePoints,San Joaquin Valley,2018-01-03,0.04,0.0,125.0,8.4,61.1,37.3,49.9,79.0,49.0,68.0,39.9,4.5,107.5,51.3


In [5]:
weather = weather[weather['Date']<=LAST_DATE]

print("Date Range: ", weather['Date'].min(), "-", weather['Date'].max())

Date Range:  2018-01-01 - 2024-12-31


In [6]:
weather = weather.drop_duplicates()
weather.duplicated().sum()

0

**initial observations:**
- The dataset contains `351,978` rows and `18` columns.
- `Date` column is currently a string object, conversion to date format is neccessary
- various columns contain `NULL` values

### 1.2 Wildfire Damage Data - `damage_cleancsv`

Provides destruction information from historical recorded wildfires that are used to construct the classification target (low, medium, high) for model training and evaluation. Relevant variables include:

- `Start` - Date fire started
- `Lat` - Latitude
- `Lon` - Longitude
- `Fire Name` - Fire Name
- `Adjusted Value` - Estimated dollar amount of damage caused

Source: [CAL FIRE – Fire Incidents](https://www.fire.ca.gov/)  
This data was extracted and structured from publicly available incident records for analytical use. See appendix A for details.

In [7]:
fire_damage = pd.read_csv("../data/raw/damage/fire_damages.csv")
basic_explore(fire_damage)

Rows:  227  Columns:  7
Duplicates  0
Total NA values:  0  of  1589 datapoints


In [8]:
fire_damage.sample(3)

,Unnamed: 0,Fire Name,Start,Estimated Damage,Lat,Lon,County
113,113,Lower,2024-08-10,629000.0,40.582292,-122.449140,Shasta
58,58,Delta,2018-08-04,10736250.0,40.978789,-122.433935,Shasta
134,134,Nixon,2024-07-29,10335100.0,33.471666,-116.746959,Riverside


In [9]:
fire_damage.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 227 entries, 0 to 226
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Unnamed: 0        227 non-null    int64  
 1   Fire Name         227 non-null    object 
 2   Start             227 non-null    object 
 3   Estimated Damage  227 non-null    float64
 4   Lat               227 non-null    float64
 5   Lon               227 non-null    float64
 6   County            227 non-null    object 
dtypes: float64(3), int64(1), object(3)
memory usage: 12.5+ KB


In [10]:
after_total = fire_damage['Estimated Damage'].values.sum()
print(f"Total Estimated Value: ${after_total:,.0f}")

Total Estimated Value: $30,852,174,450


**initial observations:**
- The dataset contains `172` rows (wildfire incidents) and `10` columns.
- Records without finish times were imputed with the mean number duration of the rest of the dataset

### 1.3 Fire Station Metadata - `i04_CIMIS_Weather_Stations.csv`

Contains geographic and descriptive information for all relevant CIMIS weather stations in California. The dataset includes spatial data. including `latitude`, `longitude`, `station ID`, `elevation`, and `regional information`. These are merged with the weather readings to analyze regional weather trends and mapping results.

Source: [California Department of Water Resources – CIMIS](https://cimis.water.ca.gov/)  
Accessed and processed in accordance with public data use policies.

In [11]:
stations = pd.read_csv("../data/raw/i04_CIMIS_Weather_Stations.csv")
basic_explore(stations)
stations.head(3)

Rows:  268  Columns:  18
Duplicates  0
Total NA values:  276  of  4824 datapoints


,X,Y,OBJECTID,Name,County,Latitude,Longitude,Status,Date_Data_Refers_To,Comments,Source,Last_Modified_Date,ID,DWR_Regional_Office_,Elevation,Connect,Disconnect,Last_Modified_By
0,-119.732000,36.814000,1,Fresno/F.S.U. USDA,Fresno,36.814000,-119.732000,Inactive,2023/02/08 00:00:00+00,NaN,"CIMIS, DWR",2023/02/09 00:00:00+00,1.0,SCRO,340,1982/06/07 00:00:00+00,9/25/1988,"AU, GDSS"
1,-120.112906,36.336222,2,FivePoints,Fresno,36.336222,-120.112906,Active,2023/02/08 00:00:00+00,NaN,"CIMIS, DWR",2023/02/09 00:00:00+00,2.0,SCRO,285,1982/06/07 00:00:00+00,Active,"AU, GDSS"
2,-121.793000,36.881000,3,Beach /Santa Cruz CO,Santa Cruz,36.881000,-121.793000,Inactive,2023/02/08 00:00:00+00,NaN,"CIMIS, DWR",2023/02/09 00:00:00+00,3.0,SCRO,10,1982/05/30 00:00:00+00,8/25/1986,"AU, GDSS"


In [12]:
stations.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 268 entries, 0 to 267
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   X                     268 non-null    float64
 1   Y                     268 non-null    float64
 2   OBJECTID              268 non-null    int64  
 3   Name                  268 non-null    object 
 4   County                268 non-null    object 
 5   Latitude              268 non-null    float64
 6   Longitude             268 non-null    float64
 7   Status                268 non-null    object 
 8   Date_Data_Refers_To   267 non-null    object 
 9   Comments              0 non-null      float64
 10  Source                267 non-null    object 
 11  Last_Modified_Date    267 non-null    object 
 12  ID                    267 non-null    float64
 13  DWR_Regional_Office_  267 non-null    object 
 14  Elevation             268 non-null    int64  
 15  Connect               2

**initial observations:**
- `OBJECTID` represents station ID to match with CIMIS data
- `County`, `Latitude`, `Longitude`, `Elevation` only variables needed from this dataset and all appear to have zero null values.
- `County` field is different case than damages dataset.

## 2. Data Cleaning and Merging

 ### 2.1 Historical Weather Merge
 Merge weather data with geographic information of CIMIS stations

In [13]:
# Merge geographic station info into weather_data
weather_station = weather.merge(
    stations[['OBJECTID', 'County', 'Latitude', 'Longitude', 'Elevation']],
    left_on='Stn Id', right_on='OBJECTID',
    how='left', indicator=True
)

In [14]:
post_merge_check(weather_station,weather)
weather_station = weather_station.drop(columns=['_merge'])
display(weather_station.sample(1))

Premerged shape:  (347375, 18)
Merged shape:  (347375, 24)
Duplicates before merge:  0
Duplicates after merge:  0
NA values before merge:  71799
NA values after merge:  71799


,Stn Id,Stn Name,CIMIS Region,Date,ETo (in),Precip (in),Sol Rad (Ly/day),Avg Vap Pres (mBars),Max Air Temp (F),Min Air Temp (F),...,Avg Rel Hum (%),Dew Point (F),Avg Wind Speed (mph),Wind Run (miles),Avg Soil Temp (F),OBJECTID,County,Latitude,Longitude,Elevation
108721,129,Pajaro,Monterey Bay,2020-11-15,0.08,0.0,306.0,9.3,69.5,36.8,...,68.0,42.6,2.8,67.2,57.1,129,Monterey,36.902778,-121.741931,65


> Merge Notes
> - Rows stayed consistent
> - No duplicates generated
> - NA values stayed consistent (to be imputed later)

---

## 3. Data Imputation

Display percentage of the amount of NA values in each column

In [15]:
NA_values = weather_station.isna().mean().round(4) * 100
NA_values = NA_values[NA_values > 0]
NA_values = NA_values[NA_values > 0].map(lambda x: f"{x:.2f}%")
NA_values

ETo (in)                5.86%
Precip (in)             0.73%
Sol Rad (Ly/day)        0.71%
Avg Vap Pres (mBars)    0.85%
Max Air Temp (F)        1.31%
Min Air Temp (F)        1.00%
Avg Air Temp (F)        0.95%
Max Rel Hum (%)         0.90%
Min Rel Hum (%)         0.90%
Avg Rel Hum (%)         1.66%
Dew Point (F)           1.66%
Avg Wind Speed (mph)    0.73%
Wind Run (miles)        0.73%
Avg Soil Temp (F)       2.68%
dtype: object

*impute_median_data* - file from src.data_utils that fills all NA values with the median of each column.

In [16]:
# Fill missing numeric weather values in the data dataframe using the median of each column.
weather_station = impute_median_data(weather_station)

In [17]:
NA_values = weather_station.isna().mean().round(4) * 100
NA_values = NA_values[NA_values > 0]
NA_values = NA_values[NA_values > 0].map(lambda x: f"{x:.2f}%")
NA_values

Series([], dtype: float64)

In [18]:
weather_station.duplicated().sum()

0

### 3.3 Identify Missing Dates
Some weather stations have gaps in daily records due to maintenance or equipment failures. Loop through all counties and compare the actual recorded dates to the expected continuous date range to identify missing entries.

**Note:** These missing dates could be imputed in future updates.

In [19]:
missing_dates = identify_missing_station_dates(weather_station)

In [20]:
print("Number of missing entries: ", len(missing_dates))

Number of missing entries:  31909


### 3.4 Spatial Join of Nearest Station Location with Fire Damage Datasets

To prepare for feature engineering, we spatially join each fire location to its nearest CIMIS weather station. This enables us to associate daily environmental conditions with each fire based on geographic proximity, rather than relying solely on county or administrative boundaries.

In [21]:
buffer_dist = 20000 

In [22]:
# Fire locations
fire_damage['geometry'] = gpd.points_from_xy(fire_damage['Lon'], fire_damage['Lat'])
fire_gdf = gpd.GeoDataFrame(fire_damage, geometry='geometry', crs='EPSG:4326')

fire_buffer = fire_gdf.copy()
fire_buffer['geometry'] = fire_buffer.buffer(buffer_dist)

# Unique station locations
station_locations = weather_station[['Stn Id', 'Latitude', 'Longitude']].drop_duplicates()
station_locations['geometry'] = gpd.points_from_xy(station_locations['Longitude'], station_locations['Latitude'])
station_gdf = gpd.GeoDataFrame(station_locations, geometry='geometry', crs='EPSG:4326')

In [23]:
# Spatial join: nearest station for each fire
fire_with_station = gpd.sjoin_nearest(
    fire_gdf,
    station_gdf[['Stn Id', 'geometry']],
    how='left',
    distance_col='station_dist'
)

In [24]:
fire_with_station

,Unnamed: 0,Fire Name,Start,Estimated Damage,Lat,Lon,County,geometry,index_right,Stn Id,station_dist
0,0,46th,2019-10-31,6.834400e+06,33.973814,-117.430053,Riverside,POINT (-117.43005 33.97381),27736,44,0.093492
1,1,Aborn,2019-07-15,7.600000e+05,37.320320,-121.754095,Santa Clara,POINT (-121.75409 37.32032),187572,191,0.367749
2,2,Aero,2024-06-17,1.201200e+06,37.992816,-120.661811,Calaveras,POINT (-120.66181 37.99282),195201,194,0.332019
3,3,Agua,2022-07-18,1.001000e+06,37.483697,-120.021368,Mariposa,POINT (-120.02137 37.48370),131715,148,0.402762
4,4,Airport,2024-09-09,9.990035e+07,33.656392,-117.443056,Orange,POINT (-117.44306 33.65639),301747,245,0.146400
...,...,...,...,...,...,...,...,...,...,...,...
222,222,Winding,2022-07-18,3.176000e+06,39.319278,-121.276206,Yuba,POINT (-121.27621 39.31928),61697,84,0.077515
223,223,Windy,2021-09-10,8.928000e+06,35.851688,-118.598383,Tulare,POINT (-118.59838 35.85169),156328,169,0.545886
224,224,Woods,2022-09-01,3.200000e+05,37.969815,-120.400392,Tuolumne,POINT (-120.40039 37.96981),195201,194,0.515830
225,225,Woolsey,2018-11-08,1.088312e+09,34.077969,-118.818547,Los Angeles,POINT (-118.81855 34.07797),243004,217,0.193524


In [25]:
fire_with_station.isna().sum()

Unnamed: 0          0
Fire Name           0
Start               0
Estimated Damage    0
Lat                 0
Lon                 0
County              0
geometry            0
index_right         0
Stn Id              0
station_dist        0
dtype: int64

In [26]:
post_merge_check(fire_with_station,fire_gdf)
fire_with_station.sample(2)

Premerged shape:  (227, 8)
Merged shape:  (227, 11)
Duplicates before merge:  0
Duplicates after merge:  0
NA values before merge:  0
NA values after merge:  0


,Unnamed: 0,Fire Name,Start,Estimated Damage,Lat,Lon,County,geometry,index_right,Stn Id,station_dist
86,86,Hesperia,2023-09-27,480000.0,35.822710,-121.050145,Monterey,POINT (-121.05014 35.82271),90206,113,0.300529
126,126,Monument,2021-08-07,11170000.0,40.718769,-123.117377,Trinity,POINT (-123.11738 40.71877),258911,224,0.811866


### 3.5 Merge Weather and Fire Damage Datasets

To prepare for feature engineering, we merge the cleaned CIMIS weather data with the fire damage records. This allows us to associate environmental conditions with known fire events, based on shared county and date information.

In [27]:
weather['Date'] = pd.to_datetime(weather['Date'])

In [28]:
weather

,Stn Id,Stn Name,CIMIS Region,Date,ETo (in),Precip (in),Sol Rad (Ly/day),Avg Vap Pres (mBars),Max Air Temp (F),Min Air Temp (F),Avg Air Temp (F),Max Rel Hum (%),Min Rel Hum (%),Avg Rel Hum (%),Dew Point (F),Avg Wind Speed (mph),Wind Run (miles),Avg Soil Temp (F)
0,2,FivePoints,San Joaquin Valley,2018-01-01,0.06,0.00,219.0,7.3,63.4,35.3,47.8,82.0,46.0,65.0,36.6,3.3,78.3,51.1
1,2,FivePoints,San Joaquin Valley,2018-01-02,0.04,0.00,127.0,7.4,59.8,37.7,47.2,80.0,52.0,67.0,36.7,3.1,74.5,51.3
2,2,FivePoints,San Joaquin Valley,2018-01-03,0.04,0.00,125.0,8.4,61.1,37.3,49.9,79.0,49.0,68.0,39.9,4.5,107.5,51.3
3,2,FivePoints,San Joaquin Valley,2018-01-04,0.07,0.01,219.0,11.6,69.2,48.7,56.8,94.0,52.0,74.0,48.5,5.8,140.2,53.0
4,2,FivePoints,San Joaquin Valley,2018-01-05,0.07,0.00,239.0,12.7,73.8,47.5,59.8,94.0,49.0,72.0,50.8,4.2,101.4,54.4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
351950,268,Nubieber,Northeast Plateau,2024-12-26,0.02,0.38,95.0,6.7,39.5,31.6,35.7,98.0,88.0,94.0,34.1,6.4,153.2,37.6
351951,268,Nubieber,Northeast Plateau,2024-12-27,0.03,0.16,122.0,8.0,46.3,37.8,42.4,94.0,75.0,87.0,38.8,9.1,219.1,38.5
351952,268,Nubieber,Northeast Plateau,2024-12-28,0.02,0.02,75.0,9.1,51.2,41.5,45.8,97.0,69.0,87.0,42.1,8.3,200.0,40.2
351953,268,Nubieber,Northeast Plateau,2024-12-29,0.02,1.02,77.0,7.0,49.3,29.6,39.4,97.0,65.0,86.0,35.4,8.9,213.4,41.0


In [29]:
summary_rows = []

for _, fire in fire_with_station.iterrows():
    stn = fire['Stn Id']
    start = pd.Timestamp(fire['Start'])  # Fire start date
    
    # Filter 7-day weather history before fire
    mask = (
        (weather['Stn Id'] == stn) &
        (weather['Date'] >= start - pd.Timedelta(days=7)) &
        (weather['Date'] < start)
    )
    
    w = weather.loc[mask]
    
    summary_rows.append({
        'Fire Name': fire['Fire Name'],
        'Start': fire['Start'],
        'Stn Id': stn,
        'Estimated Damage': fire['Estimated Damage'],
        'County': fire['County'],

        # 🔥 Weather summaries (7-day window)
        'Avg Air Temp (F)': w['Avg Air Temp (F)'].mean(),
        'Max Air Temp (F)': w['Max Air Temp (F)'].mean(),
        'Min Air Temp (F)': w['Min Air Temp (F)'].mean(),

        'Avg Rel Hum (%)': w['Avg Rel Hum (%)'].mean(),
        'Min Rel Hum (%)': w['Min Rel Hum (%)'].mean(),
        'Max Rel Hum (%)': w['Max Rel Hum (%)'].mean(),

        'Avg Wind Speed (mph)': w['Avg Wind Speed (mph)'].mean(),
        'Avg Soil Temp (F)': w['Avg Soil Temp (F)'].mean(),
        'Wind Run (miles)': w['Wind Run (miles)'].mean(),

        'Precip (in)': w['Precip (in)'].sum(),
        'Dew Point (F)': w['Dew Point (F)'].sum(),
        
        'Avg Vap Pres (mBars)': w['Avg Vap Pres (mBars)'].mean(),
        'Sol Rad (Ly/day)': w['Sol Rad (Ly/day)'].mean(),
        'ETo (in)': w['ETo (in)'].sum(),  # or .mean() depending on your preference
    })


weather_summary = pd.DataFrame(summary_rows)


In [30]:
weather_summary = weather_summary.dropna()

In [31]:
weather_summary

,Fire Name,Start,Stn Id,Estimated Damage,County,Avg Air Temp (F),Max Air Temp (F),Min Air Temp (F),Avg Rel Hum (%),Min Rel Hum (%),Max Rel Hum (%),Avg Wind Speed (mph),Avg Soil Temp (F),Wind Run (miles),Precip (in),Dew Point (F),Avg Vap Pres (mBars),Sol Rad (Ly/day),ETo (in)
0,46th,2019-10-31,44,6.834400e+06,Riverside,65.614286,79.228571,51.628571,22.857143,10.857143,45.571429,4.914286,61.142857,118.142857,0.00,161.2,4.642857,429.428571,1.13
1,Aborn,2019-07-15,191,7.600000e+05,Santa Clara,67.485714,83.557143,54.142857,63.571429,37.714286,93.857143,4.028571,72.885714,96.428571,0.00,381.7,14.542857,709.714286,1.72
2,Aero,2024-06-17,194,1.201200e+06,Calaveras,72.500000,89.300000,55.342857,60.714286,40.714286,89.571429,4.614286,68.342857,110.328571,0.00,406.7,16.857143,701.857143,1.83
4,Airport,2024-09-09,245,9.990035e+07,Orange,82.185714,101.500000,65.842857,52.857143,27.857143,85.000000,1.314286,75.828571,31.500000,0.00,437.9,19.385714,535.571429,1.39
5,Alisal,2021-10-11,64,8.800350e+06,Santa Barbara,61.742857,82.085714,47.185714,67.857143,32.857143,98.285714,2.928571,75.157143,70.057143,0.13,353.0,12.671429,408.571429,0.92
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
221,Willow,2020-09-01,84,9.017050e+06,Yuba,78.271429,91.342857,64.471429,46.285714,31.714286,67.142857,3.100000,76.442857,74.657143,0.00,391.7,15.285714,477.285714,1.39
222,Winding,2022-07-18,84,3.176000e+06,Yuba,83.471429,97.442857,68.085714,31.285714,16.428571,53.000000,4.057143,75.885714,97.314286,0.00,347.6,12.171429,715.571429,2.11
224,Woods,2022-09-01,194,3.200000e+05,Tuolumne,73.600000,90.785714,58.857143,61.000000,39.428571,88.285714,4.800000,72.042857,115.185714,0.00,415.2,17.271429,566.285714,1.50
225,Woolsey,2018-11-08,217,1.088312e+09,Los Angeles,66.228571,81.857143,54.500000,52.000000,31.142857,77.142857,3.728571,66.857143,89.871429,0.00,311.5,10.542857,320.571429,0.88


In [32]:
fires_enriched = fire_with_station.merge(
    weather_summary,
    on=['Fire Name', 'Start', 'Stn Id'],
    how='left'
)

In [33]:
fires_enriched = fires_enriched.dropna()

In [34]:
fires_enriched['Estimated Damage'] = fires_enriched['Estimated Damage_x']
fires_enriched = fires_enriched.drop(columns=['Estimated Damage_x','Estimated Damage_y'])

In [35]:
fires_enriched = fires_enriched.copy()

fires_enriched['Damage_Class'] = pd.qcut(
    fires_enriched['Estimated Damage'],
    q=3,
    labels=['Low', 'Moderate', 'High']
)

fires_enriched = fires_enriched.drop(columns=['County_y','index_right'])
fires_enriched = fires_enriched.rename(columns={'County_x': 'County'})

In [36]:
fires_enriched.isna().sum()

Unnamed: 0              0
Fire Name               0
Start                   0
Lat                     0
Lon                     0
County                  0
geometry                0
Stn Id                  0
station_dist            0
Avg Air Temp (F)        0
Max Air Temp (F)        0
Min Air Temp (F)        0
Avg Rel Hum (%)         0
Min Rel Hum (%)         0
Max Rel Hum (%)         0
Avg Wind Speed (mph)    0
Avg Soil Temp (F)       0
Wind Run (miles)        0
Precip (in)             0
Dew Point (F)           0
Avg Vap Pres (mBars)    0
Sol Rad (Ly/day)        0
ETo (in)                0
Estimated Damage        0
Damage_Class            0
dtype: int64

## 4. Suplementary Data

### 4.1 Merge population density data

Population Data obtained from 2020 US Census Data.

In [37]:
pop_density = pd.read_csv("../data/raw/pop.csv")
basic_explore(pop_density)

Rows:  58  Columns:  8
Duplicates  0
Total NA values:  0  of  464 datapoints


Basic formatting of column names and cases for merging

In [38]:
pop_density.rename(columns={'POPESTIMATE': 'Total Population'}, inplace=True)
pop_keep = pop_density.drop(columns = ['Unnamed: 0','County','YEAR','Size (square miles)','class'])
pop_keep.rename(columns={'CTYNAME': 'County'}, inplace=True)
#pop_keep['County'] = pop_keep['County'].str.upper()
pop_keep.head(2)

,County,Total Population,density
0,Alameda,1622188,2195.052908
1,Alpine,1141,1.545379


In [39]:
fires_enriched

,Unnamed: 0,Fire Name,Start,Lat,Lon,County,geometry,Stn Id,station_dist,Avg Air Temp (F),...,Avg Wind Speed (mph),Avg Soil Temp (F),Wind Run (miles),Precip (in),Dew Point (F),Avg Vap Pres (mBars),Sol Rad (Ly/day),ETo (in),Estimated Damage,Damage_Class
0,0,46th,2019-10-31,33.973814,-117.430053,Riverside,POINT (-117.43005 33.97381),44,0.093492,65.614286,...,4.914286,61.142857,118.142857,0.00,161.2,4.642857,429.428571,1.13,6.834400e+06,Moderate
1,1,Aborn,2019-07-15,37.320320,-121.754095,Santa Clara,POINT (-121.75409 37.32032),191,0.367749,67.485714,...,4.028571,72.885714,96.428571,0.00,381.7,14.542857,709.714286,1.72,7.600000e+05,Low
2,2,Aero,2024-06-17,37.992816,-120.661811,Calaveras,POINT (-120.66181 37.99282),194,0.332019,72.500000,...,4.614286,68.342857,110.328571,0.00,406.7,16.857143,701.857143,1.83,1.201200e+06,Moderate
4,4,Airport,2024-09-09,33.656392,-117.443056,Orange,POINT (-117.44306 33.65639),245,0.146400,82.185714,...,1.314286,75.828571,31.500000,0.00,437.9,19.385714,535.571429,1.39,9.990035e+07,High
5,5,Alisal,2021-10-11,34.505960,-120.083681,Santa Barbara,POINT (-120.08368 34.50596),64,0.077311,61.742857,...,2.928571,75.157143,70.057143,0.13,353.0,12.671429,408.571429,0.92,8.800350e+06,High
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
221,221,Willow,2020-09-01,39.376113,-121.326385,Yuba,POINT (-121.32639 39.37611),84,0.124016,78.271429,...,3.100000,76.442857,74.657143,0.00,391.7,15.285714,477.285714,1.39,9.017050e+06,High
222,222,Winding,2022-07-18,39.319278,-121.276206,Yuba,POINT (-121.27621 39.31928),84,0.077515,83.471429,...,4.057143,75.885714,97.314286,0.00,347.6,12.171429,715.571429,2.11,3.176000e+06,Moderate
224,224,Woods,2022-09-01,37.969815,-120.400392,Tuolumne,POINT (-120.40039 37.96981),194,0.515830,73.600000,...,4.800000,72.042857,115.185714,0.00,415.2,17.271429,566.285714,1.50,3.200000e+05,Low
225,225,Woolsey,2018-11-08,34.077969,-118.818547,Los Angeles,POINT (-118.81855 34.07797),217,0.193524,66.228571,...,3.728571,66.857143,89.871429,0.00,311.5,10.542857,320.571429,0.88,1.088312e+09,High


Merge in population data to main dataset and case study datasets.

In [40]:
fires_pop = fires_enriched.merge(
    pop_keep,
    left_on='County', right_on='County',
    how='left', indicator=True
)

In [41]:
print("Main Weather Data:")
post_merge_check(fires_pop,fires_enriched)
fires_pop = fires_pop.drop(columns=['_merge'])

Main Weather Data:
Premerged shape:  (200, 25)
Merged shape:  (200, 28)
Duplicates before merge:  0
Duplicates after merge:  0
NA values before merge:  0
NA values after merge:  0


In [42]:
fires_pop 

,Unnamed: 0,Fire Name,Start,Lat,Lon,County,geometry,Stn Id,station_dist,Avg Air Temp (F),...,Wind Run (miles),Precip (in),Dew Point (F),Avg Vap Pres (mBars),Sol Rad (Ly/day),ETo (in),Estimated Damage,Damage_Class,Total Population,density
0,0,46th,2019-10-31,33.973814,-117.430053,Riverside,POINT (-117.43005 33.97381),44,0.093492,65.614286,...,118.142857,0.00,161.2,4.642857,429.428571,1.13,6.834400e+06,Moderate,2492442,345.861225
1,1,Aborn,2019-07-15,37.320320,-121.754095,Santa Clara,POINT (-121.75409 37.32032),191,0.367749,67.485714,...,96.428571,0.00,381.7,14.542857,709.714286,1.72,7.600000e+05,Low,1877592,1455.384854
2,2,Aero,2024-06-17,37.992816,-120.661811,Calaveras,POINT (-120.66181 37.99282),194,0.332019,72.500000,...,110.328571,0.00,406.7,16.857143,701.857143,1.83,1.201200e+06,Moderate,46565,45.651513
3,4,Airport,2024-09-09,33.656392,-117.443056,Orange,POINT (-117.44306 33.65639),245,0.146400,82.185714,...,31.500000,0.00,437.9,19.385714,535.571429,1.39,9.990035e+07,High,3135755,3966.448259
4,5,Alisal,2021-10-11,34.505960,-120.083681,Santa Barbara,POINT (-120.08368 34.50596),64,0.077311,61.742857,...,70.057143,0.13,353.0,12.671429,408.571429,0.92,8.800350e+06,High,441257,161.331803
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,221,Willow,2020-09-01,39.376113,-121.326385,Yuba,POINT (-121.32639 39.37611),84,0.124016,78.271429,...,74.657143,0.00,391.7,15.285714,477.285714,1.39,9.017050e+06,High,85722,135.670423
196,222,Winding,2022-07-18,39.319278,-121.276206,Yuba,POINT (-121.27621 39.31928),84,0.077515,83.471429,...,97.314286,0.00,347.6,12.171429,715.571429,2.11,3.176000e+06,Moderate,85722,135.670423
197,224,Woods,2022-09-01,37.969815,-120.400392,Tuolumne,POINT (-120.40039 37.96981),194,0.515830,73.600000,...,115.185714,0.00,415.2,17.271429,566.285714,1.50,3.200000e+05,Low,54204,24.406542
198,225,Woolsey,2018-11-08,34.077969,-118.818547,Los Angeles,POINT (-118.81855 34.07797),217,0.193524,66.228571,...,89.871429,0.00,311.5,10.542857,320.571429,0.88,1.088312e+09,High,9663345,2381.377714


### 4.2 Merge Mean Income Data

Mean income data obtained from 2020 US Census Data.

In [43]:
mean_income = pd.read_csv("../data/raw/mean_income_by_county.csv")
basic_explore(mean_income)

Rows:  58  Columns:  2
Duplicates  0
Total NA values:  0  of  116 datapoints


format data to match case and adjust `Mean Income` column to be numerical

In [44]:
#mean_income['County'] = mean_income['County'].str.upper()
mean_income['Mean Income'] = (
    mean_income['Mean Income']
    .str.replace(',', '', regex=False)  # Remove commas
    .astype(int)                        # Convert to integer
)

mean_income.head(2)

,County,Mean Income
0,Alameda,138489
1,Alpine,107737


Merge mean income data into main dataset and case study data

In [45]:
fire_pop_income = fires_pop.merge(
    mean_income,
    left_on='County', right_on='County',
    how='left', indicator=True
)

In [46]:
print("Main Weather Data:")
post_merge_check(fire_pop_income,fires_pop)

Main Weather Data:
Premerged shape:  (200, 27)
Merged shape:  (200, 29)
Duplicates before merge:  0
Duplicates after merge:  0
NA values before merge:  0
NA values after merge:  0


In [47]:
#fire_pop_income = fire_pop_income.drop(columns='_merge')
fire_pop_income.head(2)

,Unnamed: 0,Fire Name,Start,Lat,Lon,County,geometry,Stn Id,station_dist,Avg Air Temp (F),...,Dew Point (F),Avg Vap Pres (mBars),Sol Rad (Ly/day),ETo (in),Estimated Damage,Damage_Class,Total Population,density,Mean Income,_merge
0,0,46th,2019-10-31,33.973814,-117.430053,Riverside,POINT (-117.43005 33.97381),44,0.093492,65.614286,...,161.2,4.642857,429.428571,1.13,6834400.0,Moderate,2492442,345.861225,93563,both
1,1,Aborn,2019-07-15,37.320320,-121.754095,Santa Clara,POINT (-121.75409 37.32032),191,0.367749,67.485714,...,381.7,14.542857,709.714286,1.72,760000.0,Low,1877592,1455.384854,174331,both


In [48]:
# Define the class mapping (ensure your classes are exactly spelled this way)
class_mapping = {'Low': 0, 'Moderate': 1, 'High': 2}

# Apply the mapping
fire_pop_income['Target'] = fire_pop_income['Damage_Class'].map(class_mapping)

In [49]:
drop_cols = ['geometry', '_merge']
model_fire_pop_income = fire_pop_income.drop(columns=drop_cols)

In [54]:
Raw_X = model_fire_pop_income.drop(columns=['Estimated Damage', 'Damage_Class','Target','County'])
Raw_y = model_fire_pop_income['Target']

In [55]:
Raw_y.value_counts()

Target
0    72
2    67
1    61
Name: count, dtype: int64

In [57]:
Raw_X.to_csv('../data/processed/Raw_X.csv',index=False)
Raw_y.to_csv('../data/processed/Raw_y.csv',index=False)